## 1. Importing libraries  
Imports core Python modules and third-party libraries (like pandas and typing helpers) that are used throughout the notebook.

In [382]:
import logging                                      # Python stdlib logging
import math                                         # Numeric helpers (NaN, floor, etc.)
import os                                           # Filesystem paths
import pandas as pd                                 # Excel / DataFrame handling
import sys                                          # Detect PyInstaller / executable
import warnings                                     # Suppress openpyxl / pandas warnings


from datetime import datetime                       # Report dates / timestamps
from statistics import mean                         # Simple average for peer comparisons
from dataclasses import dataclass                   # Dataclass decorator for models

from openpyxl import Workbook                       # Create XLSX from scratch
from openpyxl.styles import Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter, range_boundaries
from openpyxl.worksheet.page import PageMargins

from typing import (                                # Type hints
    Any,
    Dict,
    List,
    Literal,
    Optional,
    Tuple,
)

## 2. Setup root logger  
Configures a root `logger` to produce consistent, tagged log messages for debugging and audit trails.

In [384]:
# Setup root logger (DEBUG level, timestamp format) - shared across package
logging.basicConfig(
    level=logging.DEBUG, 
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
)
logger = logging.getLogger("sn_rating")             # Package-specific logger name


## 3. Configuring score to rating  
Defines the `SCORE_TO_RATING` mapping, which converts numeric scores into rating grades using predefined cutoffs.

In [386]:
# Score → Rating lookup: [(threshold, rating), ...] - higher score = better rating
SCORE_TO_RATING: List[Tuple[float, str]] = [
    (95, "AAA"),
    (90, "AA+"),
    (85, "AA"),
    (80, "AA-"),
    (75, "A+"),
    (70, "A"),
    (65, "A-"),
    (60, "BBB+"),
    (55, "BBB"),
    (50, "BBB-"),
    (45, "BB+"),
    (40, "BB"),
    (35, "BB-"),
    (30, "B+"),
    (25, "B"),
    (20, "B-"),
    (15, "CCC+"),
    (10, "CCC"),
    (5, "CCC-"),
    (2, "CC"),
    (0, "C"),
]

In [387]:
# Full rating scale top→bottom (used for notch adjustments, display)
RATING_SCALE = [
    "AAA",
    "AA+",
    "AA",
    "AA-",
    "A+",
    "A",
    "A-",
    "BBB+",
    "BBB",
    "BBB-",
    "BB+",
    "BB",
    "BB-",
    "B+",
    "B",
    "B-",
    "CCC+",
    "CCC",
    "CCC-",
    "CC",
    "C",
]


## 4. Defining rating weights (quant and qualitative)  
Defines `RATING_WEIGHTS` for quantitative vs qualitative blocks, allowing either fixed weights or automatic derivation based on the number of factors.

In [389]:
# Component weights (None=calculated dynamically by RatingModel)
RATING_WEIGHTS = {
    "quantitative": None,
    "qualitative": None,
}

## 5. Defining distress trigger threshold and max distress notches  
Defines `DISTRESS_BANDS` and `MAX_DISTRESS_NOTCHES`, which control when distress (e.g. low coverage ratios or Altman Z) triggers rating downgrades and how severe those downgrades can be.

In [391]:
# Distress triggers: {metric: [(threshold, notches_down), ...]} - sorted low→high
DISTRESS_BANDS = {
    "interest_coverage": [
        (0.5, -4),
        (0.8, -3),
        (1.0, -2),
    ],
    "dscr": [
        (0.8, -3),
        (0.9, -2),
        (1.0, -1),
    ],
    "altman_z": [
        (1.2, -4),
        (1.5, -3),
        (1.81, -2),
    ],
}
MAX_DISTRESS_NOTCHES = -4                          # Cap total downward adjustment

## 6. Defining qualitative scales and scores  
Defines `QUAL_SCORE_SCALE` that maps qualitative selections (e.g. 1–5) to normalized scores on a 0–100 scale.

In [393]:
# Qualitative overlay: {management_score_1-5 → score_boost_pct}
QUAL_SCORE_SCALE: Dict[int, float] = {
    5: 100.0,
    4: 75.0,
    3: 50.0,
    2: 25.0,
    1: 0.0,
}

## 7. Datamodel dataclasses  
Introduces typed containers for all key inputs and outputs used by the rating engine.

- **7.1 QuantInputs** – Holds all quantitative inputs, including current/prior financial ratios, components for Altman Z, and peer data.

In [396]:
# Quantitative inputs: 3-year financial time series + peer benchmarks
@dataclass
class QuantInputs:
    fin_t0: Dict[str, float]                           # Current year financial ratios
    fin_t1: Dict[str, float]                           # Prior year financial ratios  
    fin_t2: Dict[str, float]                           # Two years prior financial ratios
    components_t0: Dict[str, float]                    # Current year component scores
    components_t1: Dict[str, float]                    # Prior year component scores
    components_t2: Dict[str, float]                    # Two years prior component scores
    peers_t0: Dict[str, List[float]]                   # Peer group percentiles by ratio

- **7.2 QualInputs** – Holds qualitative factor values (typically 1–5) for the current period.

In [398]:
# Qualitative inputs: Management/business risk factors (2-year history)
@dataclass
class QualInputs:
    factors_t0: Dict[str, int]                         # Current qualitative scores (1-5)
    factors_t1: Dict[str, int]                         # Prior year qualitative scores (1-5)

- **7.3 RatioConfig** – Configuration for each quantitative ratio (code, name, bucket, weight).

In [400]:
# Quantitative ratio configuration (weighting, bucketing for Excel UI)
@dataclass  
class RatioConfig:
    code: str                                         # e.g. "DEBT_EBITDA"
    name: str                                         # Human-readable name
    bucket: Optional[str] = None                      # "LEVERAGE", "PROFITABILITY", etc.
    weight: Optional[float] = None                    # User override (default 1.0)

- **7.4 QualFactorConfig** – Configuration for each qualitative factor (code, name, bucket, weight).

In [402]:
# Qualitative factor configuration (bucketing for Excel UI)
@dataclass
class QualFactorConfig:
    code: str                                         # e.g. "MGMT_QUALITY"
    name: str                                         # Human-readable name  
    bucket: Optional[str] = None                      # "MANAGEMENT", "BUSINESS_RISK"
    weight: Optional[float] = None                    # User override (default 1.0)

- **7.5 RatingOutputs** – Structured container for all model outputs: scores, ratings, outlooks, flags, logs, and explanations.

In [404]:
# Complete rating outputs (Excel export + diagnostics)
@dataclass
class RatingOutputs:
    issuer_name: str                                  # Company name
    
    quantitative_score: float                         # Raw quant score (0-100)
    qualitative_score: float                          # Qual overlay score (0-100)  
    combined_score: float                             # Weighted final score (0-100)
    peer_score: Optional[float]                       # Percentile vs peers
    
    base_rating: str                                  # Score → rating (AAA, BB-, etc.)
    hardstop_rating: str                              # Distress threshold ceiling applied
    sovereign_rating: Optional[str]                   # Parent country rating
    capped_rating: str                                # Constrained rating
    final_rating: str                                 # Delivered rating
    
    hardstop_triggered: bool                          # Distressed rating thresholds breached or not
    hardstop_details: Dict[str, float]                # Details of the distressed ratio and the notches down
    distress_notches: int                             # Down notches from distress (-4 to 0)
    sovereign_cap_binding: bool                       # Rating actually constrained due to the sovereign rating?
    sovereign_outlook: Optional[str]                  # Parent country rating outlook
    outlook: str                                      # Final outlook (Stable, Negative)
    bucket_avgs: Dict[str, float]                     # Avg by bucket (Leverage=72.3)
    
    altman_z_t0: float                                # Bankruptcy score
    flags: Dict[str, bool]                            # Distress flags triggered
    
    rating_explanation: str                           # Human-readable methodology
    peer_underperform_count: int                      # # peers beaten
    peer_outperform_count: int                        # # peers worse  
    peer_on_par_count: int                            # # within +/-10% of peer avg
    peer_total_compared: int                          # Total peers benchmarked
    
    band_position: str                                # "Top quartile", "Distressed"
    
    ratio_log: List[Dict[str, object]]                # All ratio calculations
    qual_log: List[Dict[str, object]]                 # All qual calculations
    
    base_outlook: str                                 # Pre-adjustment outlook
    hardstop_outlook: str                             # Sovereign-adjusted outlook  
    final_outlook: str                                # Delivered outlook
    
    n_quant: int                                      # # quantitative ratios used
    n_qual: int                                       # # qualitative factors used
    wq: float                                         # Final quant weight
    wl: float                                         # Final qual weight

## 8. Helpers  
Provides reusable utility functions and small classes used by the model for configuration, paths, scoring, and rating logic.

- **8.1 normalize_ratio_config** – Ensures each `RatioConfig` has a non-null weight and a default `"OTHERS"` bucket.

In [407]:
def normalize_ratio_config(cfg: RatioConfig) -> RatioConfig:
    """Fill in default bucket/weight for a quantitative ratio config."""
    
    weight = cfg.weight if cfg.weight is not None else 1.0     # Default weight=1.0
    bucket = cfg.bucket if cfg.bucket not in (None, "", " ") else "OTHERS"  # Fallback bucket
    return RatioConfig(                                        # Return normalized copy
        code=cfg.code,
        name=cfg.name,
        bucket=bucket,
        weight=weight,
    )

- **8.2 normalize_qual_config** – Ensures each `QualFactorConfig` has a non-null weight and a default `"OTHERS"` bucket.

In [409]:
def normalize_qual_config(cfg: QualFactorConfig) -> QualFactorConfig:
    """Fill in default bucket/weight for a qualitative factor config."""
    
    weight = cfg.weight if cfg.weight is not None else 1.0     # Default weight=1.0
    bucket = cfg.bucket if cfg.bucket not in (None, "", " ") else "OTHERS"
    return QualFactorConfig(
        code=cfg.code,
        name=cfg.name,
        bucket=bucket,
        weight=weight,
    )

- **8.3 project_root** – Locates the project root, working both in plain Python and PyInstaller EXE mode.

In [411]:
def project_root() -> str:
    """Return folder where run_sn_rating.exe / run_sn_rating.bat reside."""
    
    if hasattr(sys, "frozen"):                                 # Running as PyInstaller EXE
        base = os.path.dirname(sys.executable)                 # Use exe directory
    else:                                                      # Running as plain Python
        base = os.path.dirname(os.path.abspath(__file__))      # This file's dir: .../sn_rating
        base = os.path.dirname(base)                           # Go one level up: project root
    return base


- **8.4 input_dir** – Returns the `input` folder path under the project root, creating it if necessary.

In [413]:
def input_dir() -> str:
    """Ensure and return input/ folder under project root."""
    
    root = project_root()                                      # Base folder
    path = os.path.join(root, "input")                         # .../input
    os.makedirs(path, exist_ok=True)                           # Create if missing
    return path


- **8.5 output_dir** – Returns the `output` folder path under the project root, creating it if necessary.

In [415]:
def output_dir() -> str:
    """Ensure and return output/ folder under project root."""
    
    root = project_root()
    path = os.path.join(root, "output")
    os.makedirs(path, exist_ok=True)
    return path


- **8.6 input_path** – Builds a full path inside the `input` directory for a given filename.

In [417]:
def input_path(name: str) -> str:
    """Build full path to a file inside input/."""
    
    return os.path.join(input_dir(), name)

- **8.7 output_path** – Builds a full path inside the `output` directory for a given filename.

In [419]:
def output_path(name: str) -> str:
    """Build full path to a file inside output/."""
    
    return os.path.join(output_dir(), name)

- **8.8 resource_path** – Resolves a resource path relative to the module/executable location.

In [421]:
def resource_path(relative_path: str) -> str:
    """Locate resource relative to exe or package folder (for configs, templates, etc.)."""
    
    if getattr(sys, "frozen", False):                          # PyInstaller frozen mode
        base = os.path.dirname(sys.executable)
    else:
        base = os.path.dirname(os.path.abspath(__file__))      # Module directory
    return os.path.join(base, relative_path)


- **8.9 BandConfig** – Loads ratio band tables from Excel, normalizes them, and infers each ratio’s family and direction (`higher`/`lower`), and provides band-based scoring helpers.
    - **8.9.1 _norm_cols** – Normalizes column names and key text fields (lowercase, stripped) in the band tables.
    - **8.9.2 _load** – Reads the `lower_better`/`higher_better` sheets, builds family and direction maps, and logs them.
    - **8.9.3 get_direction** – Returns the stored direction (`higher`/`lower`) for a metric, if available. 
    - **8.9.4 lookup** – Looks up a numeric value in the configured bands and returns the corresponding band score (with clamping outside ranges). 

In [423]:
class BandConfig:
    """
    Reads sn_rating_config.xlsx and infers, for each ratio_name:
    - ratio_family (cluster) from the band rows
    - direction: 'lower' or 'higher' depending on which sheet it appears in
    """

    def __init__(self, config_path: str = "sn_rating_config.xlsx"):
        # Always resolve via input_path so exe uses its own folder
        self.config_path = input_path(config_path)             # Input config Excel path
        self.lower: Optional[pd.DataFrame] = None              # Bands where lower is better
        self.higher: Optional[pd.DataFrame] = None             # Bands where higher is better
        self.ratio_family: Dict[str, str] = {}                 # "ratio_name" → "family"
        self.direction: Dict[str, str] = {}                    # "ratio_name" → "lower"/"higher"
        self._load()                                           # Load immediately on init

    def _norm_cols(self, df: pd.DataFrame) -> pd.DataFrame:
        """Normalize column and key text to lowercase/stripped for robust matching."""
        
        df = df.copy()
        df.columns = [str(c).strip().lower() for c in df.columns]  # Standardize headers
        for c in ["ratio_family", "ratio_name"]:
            if c in df.columns:
                df[c] = df[c].astype(str).str.strip().str.lower()  # Normalize keys
        return df

    def _load(self) -> None:
        """Read Excel bands, split into lower_better/higher_better, build indexes."""
        
        with pd.ExcelFile(self.config_path) as xls:            # Open config workbook
            lb = pd.read_excel(xls, "lower_better")            # Bands: lower values are better
            hb = pd.read_excel(xls, "higher_better")           # Bands: higher values are better
        self.lower = self._norm_cols(lb)                       # Normalize text
        self.higher = self._norm_cols(hb)

        all_bands = []                                         # Collect both directions
        if self.lower is not None:
            tmp = self.lower.copy()
            tmp["__direction__"] = "lower"                     # Mark rows from lower sheet
            all_bands.append(tmp)
        if self.higher is not None:
            tmp = self.higher.copy()
            tmp["__direction__"] = "higher"                    # Mark rows from higher sheet
            all_bands.append(tmp)
        if not all_bands:                                      # Nothing loaded → bail
            return

        all_bands_df = pd.concat(all_bands, ignore_index=True) # Unified table of all bands

        for ratio_name, sub in all_bands_df.groupby("ratio_name"):  # For each metric
            fam = str(sub["ratio_family"].iloc[0])             # First family's name
            dirn = str(sub["__direction__"].iloc[0])           # "lower" or "higher"
            self.ratio_family[ratio_name] = fam
            self.direction[ratio_name] = dirn

        logger.info(                                           # Log summary for debugging
            "BandConfig: loaded families/directions for ratios: %s",
            self.ratio_family,
        )

    def get_direction(self, metric: str) -> Optional[str]:
        """Return 'lower' or 'higher' for a given ratio metric, if known."""
        
        m = metric.strip().lower()                             # Normalize lookup key
        return self.direction.get(m)

    def lookup(self, metric: str, val: float) -> Optional[float]:
        """Lookup band score for a metric/value using configured bands."""
        
        metric = metric.strip().lower()
        if val is None or (isinstance(val, float) and math.isnan(val)):
            return None                                        # No score for missing value

        dirn = self.get_direction(metric)                      # "lower"/"higher"/None
        if dirn is None:
            return None

        df = self.lower if dirn == "lower" else self.higher    # Pick proper band table
        if df is None:
            return None

        dfm = df[df["ratio_name"] == metric]                   # Filter rows for this metric
        if dfm.empty:
            return None

        # First try to find a band where min_value <= val < max_value
        for _, row in dfm.iterrows():
            lo, hi = row["min_value"], row["max_value"]
            s = row["score"]
            if lo <= val < hi:
                return float(s)                                # Found matching band

        # If outside configured bands, clamp to lowest or highest band
        lo_min = dfm["min_value"].min()
        hi_max = dfm["max_value"].max()
        if val < lo_min:
            row = dfm.loc[dfm["min_value"].idxmin()]           # Use lowest band
        else:
            row = dfm.loc[dfm["max_value"].idxmax()]           # Use highest band
        return float(row["score"])

- **8.10 classify_peer_with_bandconfig** – For a single metric, uses `BandConfig.get_direction` and a ±band around the peer average to classify the metric as under/over/on_par, returning bounds and the peer average.

In [425]:
def classify_peer_with_bandconfig(
    metric_name: str,
    value: float,
    peer_avg: float,
    band_cfg,
    band: float = 0.10,
):
    """
    Direction-aware peer classification using BandConfig:
    - direction: 'higher' or 'lower' from band_cfg.get_direction(...)
    - band: +/- % around peer_avg that counts as 'on_par'
    """
    if (
        peer_avg is None or value is None
        or (isinstance(peer_avg, float) and math.isnan(peer_avg))
        or (isinstance(value, float) and math.isnan(value))
    ):
        return None, None, None, None

    dirn = band_cfg.get_direction(metric_name)   # "higher" / "lower" / None
    if dirn is None:
        dirn = "higher"                          # Default assume 'higher is better'

    lower = peer_avg * (1 - band)               # Lower bound for on_par zone
    upper = peer_avg * (1 + band)               # Upper bound for on_par zone

    if lower <= value <= upper:
        flag = "on_par"                          # Within ±band → on par with peers
    else:
        if dirn == "higher":                     # For higher-better metrics
            flag = "over" if value > upper else "under"
        else:                                    # For lower-better metrics (e.g. leverage)
            flag = "over" if value < lower else "under"

    return lower, upper, flag, peer_avg          # Return band bounds + classification


- **8.11 score_qual_factor_numeric** – Converts a qualitative numeric selection into a score using `QUAL_SCORE_SCALE`.
 

In [427]:
def score_qual_factor_numeric(value: int) -> Optional[float]:
    """Map 1–5 qualitative score to numeric via QUAL_SCORE_SCALE."""
    
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    return QUAL_SCORE_SCALE.get(int(value))


- **8.12 compute_altman_z_from_components** – Computes the Altman Z-score from its component financial inputs.


In [429]:
def compute_altman_z_from_components(
    working_capital: float,
    total_assets: float,
    retained_earnings: float,
    ebit: float,
    market_value_equity: float,
    total_liabilities: float,
    sales: float,
) -> float:
    """Compute Altman Z-score from raw balance sheet/income statement components."""
    
    if total_assets == 0 or total_liabilities == 0:            # Avoid division by zero
        return float("nan")
    A = working_capital / total_assets
    B = retained_earnings / total_assets
    C = ebit / total_assets
    D = market_value_equity / total_liabilities
    E = sales / total_assets
    return 1.2 * A + 1.4 * B + 3.3 * C + 0.6 * D + 1.0 * E     # Standard Altman Z formula


- **8.13 compute_peer_score** – Compares a company’s ratios against peer averages and turns under/over performance into a single peer score plus counts.

In [431]:
def compute_peer_score(
    fin_current: Dict[str, float],
    peers: Dict[str, List[float]],
    band_cfg,   # BandConfig instance
    band: float = 0.10,
) -> Tuple[Optional[float], int, int, int, int]:
    """
    Compare issuer current ratios vs peer averages, return:
    (peer_score 0–100, underperform_count, outperform_count, on_par_count, total_compared).

    - direction comes from band_cfg.get_direction(rname): 'higher' or 'lower'
    - band: +/- % around peer_avg that counts as 'on_par'
    """

    under = 0
    over = 0
    on_par = 0
    total = 0

    for rname, peer_vals in peers.items():
        if rname not in fin_current or not peer_vals:
            continue

        cp = fin_current[rname]
        peer_avg = mean(peer_vals)
        if peer_avg == 0:
            continue

        total += 1

        # 10% band for "on_par"
        lower = peer_avg * (1 - band)
        upper = peer_avg * (1 + band)

        # Look up direction from BandConfig instead of guessing from the name
        dirn = band_cfg.get_direction(rname)   # "higher" / "lower" / None

        if dirn is None:
            # Safer: skip metrics with unknown direction
            # (or log a warning instead of silently assuming)
            total -= 1          # undo the increment; we are not comparing this one
            continue

        if lower <= cp <= upper:
            on_par += 1
        else:
            if dirn == "higher":
                # Higher is better
                if cp < lower:
                    under += 1
                elif cp > upper:
                    over += 1
            else:
                # Lower is better
                if cp > upper:
                    under += 1
                elif cp < lower:
                    over += 1

    if total == 0:
        return None, 0, 0, 0, 0

    under_share = under / total

    if under_share <= 0.10:
        score = 100.0
    elif under_share <= 0.30:
        score = 75.0
    elif under_share <= 0.60:
        score = 50.0
    elif under_share <= 0.80:
        score = 25.0
    else:
        score = 0.0

    return score, under, over, on_par, total


- **8.14 score_to_rating** – Maps a numeric score to a rating grade using `SCORE_TO_RATING` cutoffs.

In [433]:
def score_to_rating(score: float) -> str:
    """Map numeric score to rating using SCORE_TO_RATING cutoffs."""
    
    for cutoff, grade in SCORE_TO_RATING:                      # Iterate high→low thresholds
        if score >= cutoff:
            return grade
    raise ValueError(f"Score {score} did not match any cutoff") # only a backstop although the scenario is rare


- **8.15 safe_score_to_rating** – Wraps `score_to_rating`, returning `"N/R"` and logging an error instead of raising on invalid scores.

In [435]:
def safe_score_to_rating(score: float) -> str:
    """Wrapper that logs errors and returns 'N/R' if mapping fails."""
    
    try:
        return score_to_rating(score)
    except ValueError as e:
        logger.error("Score-to-rating mapping failed: %s", e)
        return "N/R"


- **8.16 move_notches** – Moves a rating up or down a number of notches within `RATING_SCALE`, with clamping at the ends.

In [437]:
def move_notches(grade: str, notches: int) -> str:
    """Move rating up/down by N notches within RATING_SCALE."""
    
    if grade not in RATING_SCALE:
        return grade                                           # Unknown grade → unchanged
    idx = RATING_SCALE.index(grade)                            # Current position
    new_idx = max(0, min(idx - notches, len(RATING_SCALE) - 1))# Subtract notches, clamp to bounds (ensures it is not below 0 or above the last index)
    return RATING_SCALE[new_idx]


- **8.17 apply_sovereign_cap** – Applies a sovereign ceiling by capping an issuer’s rating at the sovereign’s level using `RATING_SCALE`.

In [439]:
def apply_sovereign_cap(
    issuer_grade: str,
    sovereign_grade: Optional[str],
) -> str:
    """Apply sovereign rating cap: issuer cannot be stronger than sovereign."""
    
    if sovereign_grade is None:
        return issuer_grade
    if issuer_grade not in RATING_SCALE or sovereign_grade not in RATING_SCALE:
        return issuer_grade
    i = RATING_SCALE.index(issuer_grade)
    s = RATING_SCALE.index(sovereign_grade)
    return RATING_SCALE[max(i, s)]                             # Weaker of the two (higher index)


- **8.18 compute_effective_weights** – Determines the effective quantitative vs qualitative weights, using configured weights or ratio-based fallback.

In [441]:
def compute_effective_weights(n_quant: int, n_qual: int) -> Tuple[float, float]:
    """
    Compute final quant / qual weights:
    - If both set in RATING_WEIGHTS: use them.
    - Otherwise, allocate by count of metrics (n_quant vs n_qual).
    """
    
    wq_cfg = RATING_WEIGHTS["quantitative"]
    wl_cfg = RATING_WEIGHTS["qualitative"]

    if wq_cfg is not None and wl_cfg is not None:              # Explicit config from Excel
        return float(wq_cfg), float(wl_cfg)

    n_quant = max(n_quant, 0)
    n_qual = max(n_qual, 0)
    total = n_quant + n_qual

    if total == 0:
        return 0.0, 0.0                                        # No metrics at all

    wq = n_quant / total                                       # Proportional weight
    wl = n_qual / total
    return wq, wl

- **8.19 get_rating_band** – Returns the numeric score band (min, max) corresponding to a given rating grade. 

In [443]:
def get_rating_band(rating: str) -> Tuple[float, float]:
    """Return [min_score, max_score] band for a given rating grade."""
    
    for i, (cutoff, grade) in enumerate(SCORE_TO_RATING):
        if grade == rating:
            band_min = cutoff                                  # Lower bound = its own cutoff
            if i == 0:
                band_max = 100.0                               # Top rating extends to 100
            else:
                prev_cutoff, _ = SCORE_TO_RATING[i - 1]        # Next better rating's cutoff
                band_max = prev_cutoff - 1.0                   # One point below that
            return band_min, band_max
    raise ValueError(f"Unknown rating grade: {rating!r}")


- **8.20 derive_outlook_band_only** – Derives outlook and band position (upper/middle/lower) from combined score vs its rating band.

In [445]:
def derive_outlook_band_only(combined_score: float, rating: str) -> Tuple[str, str]:
    """
    Derive outlook ('Positive', 'Stable', 'Negative') and band position
    based only on where combined_score sits within the rating band.
    """
    
    band_min, band_max = get_rating_band(rating)
    cs = math.floor(combined_score)                            # Discrete score for logic

    if cs == band_max:
        outlook = "Positive"
    elif cs == band_min:
        outlook = "Negative"
    else:
        outlook = "Stable"

    if cs == band_min:
        band_position = "lower_band"
    elif cs == band_max:
        band_position = "upper_band"
    else:
        band_position = "middle_band"

    return outlook, band_position



- **8.21 derive_outlook_with_distress_trend** – Adjusts outlook based on distress notches and trends in key distress ratios.

In [447]:
def derive_outlook_with_distress_trend(
    base_outlook: str,
    distress_notches: int,
    fin_t0: dict[str, float],
    fin_t1: dict[str, float],
) -> str:
    """
    Adjust outlook using distress trend:
    - If distress_notches < 0 and metrics deteriorating → 'Negative'
    - Else → 'Stable' (or keep base_outlook if no distress).
    """
    
    if distress_notches >= 0:
        return base_outlook                                     # No negative distress → keep

    ratios = ["interest_coverage", "dscr", "altman_z"]          # Key distress metrics
    improving = False
    deteriorating = False

    for r in ratios:
        v0 = fin_t0.get(r)
        v1 = fin_t1.get(r)
        if v0 is None or v1 is None:
            continue
        if v0 > v1:
            improving = True                                    # Higher now than last year
        elif v0 < v1:
            deteriorating = True

    if deteriorating and not improving:
        return "Negative"
    return "Stable"


- **8.22 rating_index** – Returns the index of a rating grade in `RATING_SCALE`, or `None` if unknown.

In [449]:
def rating_index(grade: str) -> Optional[int]:
    """Return index of grade in RATING_SCALE, or None if unknown."""
    
    if grade not in RATING_SCALE:
        return None
    return RATING_SCALE.index(grade)

- **8.23 is_stronger** – Checks if rating `r1` is stronger (better) than `r2` using `RATING_SCALE` ordering. 

In [451]:
def is_stronger(r1: str, r2: str) -> bool:
    """Return True if r1 is stronger (better rating) than r2."""
    
    i1 = rating_index(r1)
    i2 = rating_index(r2)
    if i1 is None or i2 is None:
        return False
    return i1 < i2                                             # Lower index = stronger


- **8.24 is_weaker_or_equal** – Checks if rating `r1` is weaker than or equal to `r2` using `RATING_SCALE` ordering.

In [453]:
def is_weaker_or_equal(r1: str, r2: str) -> bool:
    """Return True if r1 is weaker than or equal to r2."""
    
    i1 = rating_index(r1)
    i2 = rating_index(r2)
    if i1 is None or i2 is None:
        return False
    return i1 >= i2                                            # Higher index = weaker/equal


## 9. Excel I/O converters  
Provides small utilities that convert Excel-/DataFrame-based inputs into plain Python dictionaries used by the rating model.

- **9.1 load_metadata_excel** – Reads the main Excel input file (e.g. metadata, ratios, peers, qualitative sheets) into pandas DataFrames ready for further conversion.

In [456]:
def load_metadata_excel(filename: str = "sn_rating_input.xlsx") -> Dict[str, object]:
    """Read 'metadata' sheet, update RATING_WEIGHTS, and return core meta fields."""
    
    path = resource_path(filename)                      # Get absolute path (works in .py and .exe)

    with pd.ExcelFile(path) as xls:                     # Open workbook once, re-use handle
        df = pd.read_excel(xls, sheet_name="metadata")  # Read 'metadata' sheet into DataFrame
    
    meta = dict(zip(df["field"], df["value"]))          # Turn two columns into {field: value} dict

    # Update global rating weights from metadata (user-configurable in Excel)
    q_w = meta.get("quantitative_weight")              # Read quant weight (may be None/NaN)
    l_w = meta.get("qualitative_weight")               # Read qual weight
    
    RATING_WEIGHTS["quantitative"] = (                 # Store numeric or None in config
        float(q_w) if pd.notna(q_w) else None
    )
    RATING_WEIGHTS["qualitative"] = (                  # Same for qualitative side
        float(l_w) if pd.notna(l_w) else None
    )

    def flag(name: str, default: str = "TRUE") -> bool:
        """Parse boolean-like metadata values: TRUE/YES/1 → True."""
        raw = meta.get(name, default)                  # Get value or default string
        s = str(raw).strip().upper()                   # Normalize whitespace and case
        return s in ("TRUE", "1", "YES", "Y")          # Map common truthy strings to True

    return {                                           # Build clean, typed metadata dict
        "name": str(meta.get("name", "")).strip(),
        "country": str(meta.get("country", "")).strip(),
        "id": str(meta.get("id", "")).strip(),
        "sovereign_rating": str(meta.get("sovereign_rating", "")).strip().upper(),
        "sovereign_outlook": str(meta.get("sovereign_outlook", "")).strip(),
        "enable_peer_positioning": flag("enable_peer_positioning", "FALSE"),    # when empty, returns false as flag value
        "enable_hardstops": flag("enable_hardstops", "FALSE"),                  # when empty, returns false as flag value
        "enable_sovereign_cap": flag("enable_sovereign_cap", "FALSE"),          # when empty, returns false as flag value
    }


- **9.2 _infer_time_labels** – Infers the most recent and prior period labels `[t0, t1, t2]` from a ratios DataFrame’s column names, ignoring `Unnamed` columns.

In [458]:
def _infer_time_labels(df: pd.DataFrame, drop_cols: List[str] = None) -> List[str]:
    """
    Infer [t0, t1, t2] labels from DataFrame columns,
    excluding drop_cols and any 'Unnamed' columns.
    """
    if drop_cols is None:
        drop_cols = ["metric", "weight"]           # Default for fin_ratios layout

    cols = [
        c
        for c in df.columns
        if str(c).strip().lower() not in {*(s.lower() for s in drop_cols)}
        and not str(c).startswith("Unnamed")
    ]                                              # Keeps only actual period columns

    if len(cols) < 2:
        raise ValueError("Need at least two period columns")

    t0 = cols[0]                                   # Most recent period
    t1 = cols[1]                                   # Previous period
    t2 = cols[2] if len(cols) >= 3 else cols[1]    # Fallback: reuse t1 if no t2

    return [str(t0), str(t1), str(t2)]


- **9.3 df_row_to_dict** – Converts a single-row (or keyed) DataFrame of numeric fields into a `Dict[str, float]` for ratio dictionaries such as `fin_t0` or `fin_t1`.

In [460]:
def df_row_to_dict(df: pd.DataFrame, col: str) -> Dict[str, float]:
    """Convert one numeric column into {row_index_as_str: float_value}."""
    
    if col not in df.columns:                 # If requested column missing, return empty
        return {}
    
    s = df[col]                               # Extract Series for the column
    out: Dict[str, float] = {}               # Output mapping: "ratio_code" → value
    
    for idx, v in s.items():                 # Iterate over index + value pairs
        try:
            if v is None or (isinstance(v, float) and math.isnan(v)):
                continue                     # Skip empty / NaN cells
        except TypeError:
            pass                             # Non-float values: skip NaN check, still allow cast
        out[str(idx)] = float(v)             # Store as float, key as string
    return out


- **9.4 df_qual_to_dict** – Converts a qualitative factors DataFrame into a `Dict[str, int]` mapping factor names to their selected scale values (e.g. 1–5).

In [462]:
def df_qual_to_dict(df: pd.DataFrame, col: str) -> Dict[str, int]:
    """Convert one qualitative column into {row_index_as_str: int_score}."""
    
    if col not in df.columns:                # Missing column → empty dict
        return {}
    
    s = df[col]                              # Extract Series
    out: Dict[str, int] = {}                # Output mapping: "factor_code" → int score
    
    for idx, v in s.items():                # Iterate over index + value
        try:
            if v is None or (isinstance(v, float) and math.isnan(v)):
                continue                    # Skip blanks / NaNs
        except TypeError:
            pass
        out[str(idx)] = int(v)              # Store as int, key as string
    return out

- **9.5 components_col_to_dict** – Extracts balance-sheet/P&L components from a DataFrame column and returns a `Dict[str, float]` used to compute Altman Z and other composite metrics.

In [464]:
def components_col_to_dict(df: pd.DataFrame, col: str) -> Dict[str, float]:
    """Thin wrapper for component columns, kept for semantic clarity."""
    
    return df_row_to_dict(df, col)           # Reuse numeric conversion logic


- **9.6 peers_df_to_dict** – Transforms a peers DataFrame into a `Dict[str, List[float]]` mapping each metric name to the list of peer values for that metric. 

In [466]:
def peers_df_to_dict(df_peers: pd.DataFrame) -> Dict[str, List[float]]:
    """Convert peer percentile table into {metric_code: [values...]}."""
    
    peers: Dict[str, List[float]] = {}      # Output mapping: "metric" → list of floats
    
    for metric, row in df_peers.iterrows(): # Loop each row; index is metric name/code
        vals: List[float] = []              # List of non-empty peer values
        
        for v in row.tolist():              # Iterate raw cell values across the row
            try:
                if v is None or (isinstance(v, float) and math.isnan(v)):
                    continue                # Skip missing / NaN peers
            except TypeError:
                pass
            vals.append(float(v))           # Append numeric peer value
        
        if not vals:                        # If all values empty, skip this metric
            continue
        
        peers[str(metric)] = vals           # Store list under string metric key
    
    return peers

## 10. RatingModel: core rating engine  
Implements the end-to-end rating engine that combines quantitative metrics, qualitative factors, distress constraints, and sovereign cap logic into a final rating, outlook, and detailed logs.

- **10.1 _ensure_altman_z** – Ensures an Altman Z-score is available by either using a provided value or computing it from financial components.  
- **10.2 compute_quantitative** – Computes the weighted quantitative score, logs per-ratio details, and optionally adds direction-aware peer positioning and a peer score. 
- **10.3 compute_qualitative** – Converts qualitative factors into scores, applies weights and buckets, and returns a weighted qualitative score and log.  
- **10.4 compute_distress_notches** – Applies distress triggers (interest coverage, DSCR, Altman Z) to derive downgrade notches and per-metric distress details, with a cap on maximum severity.
- **10.5 compute_final_rating** – Orchestrates the full workflow: combines quant and qual scores, maps to a base rating and outlook, applies distress hardstops and sovereign cap, finalizes rating and outlook, sets flags, composes a human-readable explanation, and returns a fully populated `RatingOutputs`.

In [468]:
class RatingModel:
    """Main engine: compute full rating from quantitative & qualitative inputs."""

    def __init__(self, cp_name: str, bands: BandConfig):
        self.cp_name = cp_name                  # Issuer / counterparty name
        self.bands = bands                      # BandConfig instance (ratio scoring tables)

    def _ensure_altman_z(self, fin: Dict[str, float], comps: Dict[str, float]) -> Optional[float]:
        """Ensure fin['altman_z'] exists; compute from components if possible."""
        
        # 1. If already present and finite, just return it
        z_existing = fin.get("altman_z")
        if z_existing is not None:
            try:
                if not math.isnan(z_existing):
                    return z_existing
            except TypeError:
                return z_existing  # Non-float but present → trust it
        
        # 2. Safely extract components
        wc   = comps.get("working_capital")
        ta   = comps.get("total_assets")
        re   = comps.get("retained_earnings")
        ebit = comps.get("ebit")
        mve  = comps.get("market_value_equity")
        tl   = comps.get("total_liabilities")
        sales = comps.get("sales")
        
        # 3. If anything essential is missing or zero, skip Altman
        if None in (wc, ta, re, ebit, mve, tl, sales) or ta == 0 or tl == 0:
            logger.info("%s-Altman_Z: skipped (missing/invalid components)", self.cp_name)
            return None
        
        # 4. Compute Z from valid components
        z = compute_altman_z_from_components(wc, ta, re, ebit, mve, tl, sales)
        fin["altman_z"] = z
        logger.info("%s-AltmanZ: computed z=%.3f from components", self.cp_name, z)
        return z


    # --------- QUANTITATIVE BLOCK ---------

    def compute_quantitative(
        self,
        q: QuantInputs,
        ratio_weights: Dict[str, float],
        enable_peer_positioning: bool,
        fin: Dict[str, float],
    ) -> Tuple[
        float,                    # quantitative_score
        Optional[float],          # peer_score
        Dict[str, float],         # bucket_avgs
        int,                      # n_quant_items
        int,                      # peer_under
        int,                      # peer_over
        int,                      # peer_on_par
        int,                      # peer_total
        List[Dict[str, object]],  # ratio_log
    ]:

        total_weighted = 0.0                              # Σ (weight * score)
        total_weight = 0.0                                # Σ weight
        bucket_weighted: Dict[str, float] = {}           # Per-bucket Σ(weight * score)
        bucket_weight: Dict[str, float] = {}             # Per-bucket Σ(weight)
        ratio_log: List[Dict[str, object]] = []          # Detailed per-ratio log
        n_quant_items = 0                                 # Count of quant metrics used

        # Per-ratio scoring
        for rname, val in fin.items():
            if self.bands.get_direction(rname) is None:  # Skip ratios not configured in bands
                continue

            score = self.bands.lookup(rname, val)        # Map value → 0-100 band score
            if score is None:
                logger.info("%s-Quant: no band/score for ratio %s", self.cp_name, rname)
                continue

            w = float(ratio_weights.get(rname, 1.0))     # Default weight=1.0 if not provided
            n_quant_items += 1

            fam = self.bands.ratio_family.get(rname.strip().lower(), "OTHERS")  # Bucket

            total_weighted += w * score                  # Aggregate totals
            total_weight += w

            bucket_weighted[fam] = bucket_weighted.get(fam, 0.0) + w * score
            bucket_weight[fam] = bucket_weight.get(fam, 0.0) + w

            logger.info(
                "%s-Quant: %s value=%.2f score=%.1f weight=%.1f family=%s",
                self.cp_name,
                rname,
                val,
                score,
                w,
                fam,
            )

            # Per-ratio peer positioning with ±10% band, direction-aware
            peer_flag = None      # under, over, on_par
            peer_avg = None
            lower_bound = None
            upper_bound = None

            if enable_peer_positioning and q.peers_t0:
                peer_vals = q.peers_t0.get(rname)
                if peer_vals:
                    try:
                        vals = [v for v in peer_vals if v is not None]
                        peer_avg = sum(vals) / len(vals) if vals else None
                    except TypeError:
                        peer_avg = None

                    if peer_avg is not None and peer_avg != 0:
                        lower_bound, upper_bound, peer_flag, peer_avg = classify_peer_with_bandconfig(
                            rname,
                            val,
                            peer_avg,
                            self.bands,
                            band=0.10,  # ±10% on-par band
                        )

            ratio_log.append(
                {
                    "Name": rname,
                    "Value": val,
                    "Score": score,
                    "Weight": w,
                    "Bucket": fam,
                    "PeerFlag": peer_flag,
                    "PeerAvg": peer_avg,
                    "PeerLowerBound": lower_bound,
                    "PeerUpperBound": upper_bound,
                    # "DistressNotches" added later
                }
            )

        # Aggregate peer score (overall peer positioning)
        peer_score: Optional[float] = None
        peer_under = peer_over = peer_on_par = peer_total = 0

        if enable_peer_positioning and q.peers_t0:
            (
                peer_score,
                peer_under,
                peer_over,
                peer_on_par,
                peer_total,
            ) = compute_peer_score(fin, q.peers_t0, self.bands, band=0.10)
                
            if peer_score is not None:
                w_peer = 1.0           #weight of the peer‑positioning component within the quantitative score block
                n_quant_items += 1
        
                total_weighted += w_peer * peer_score
                total_weight += w_peer
        
                bucket_weighted["peer"] = bucket_weighted.get("peer", 0.0) + w_peer * peer_score
                bucket_weight["peer"] = bucket_weight.get("peer", 0.0) + w_peer
        
                logger.info(
                    "%s-PeerPositioning: score=%.1f under=%d over=%d on_par=%d total=%d",
                    self.cp_name,
                    peer_score,
                    peer_under,
                    peer_over,
                    peer_on_par,
                    peer_total,
                )
         
        else:
            peer_score = None
            peer_under = peer_over = peer_on_par = peer_total = 0


        quantitative_score = total_weighted / total_weight if total_weight > 0 else 0.0
        logger.info("%s-Quant: aggregate weighted score=%.1f", self.cp_name, quantitative_score)

        bucket_avgs = {
            b: round(bucket_weighted[b] / bucket_weight[b], 1)
            for b in bucket_weight
            if bucket_weight[b] > 0
        }

        return (
            quantitative_score,
            peer_score,
            bucket_avgs,
            n_quant_items,
            peer_under,
            peer_over,
            peer_on_par,
            peer_total,
            ratio_log,
        )

    # --------- QUALITATIVE BLOCK ---------

    def compute_qualitative(
        self,
        ql: QualInputs,
        qual_weights: Dict[str, float],
        qual_buckets: Dict[str, str],
    ) -> Tuple[float, int, List[Dict[str, object]]]:
        """Compute weighted qualitative score and log."""
        
        total_weighted = 0.0
        total_weight = 0.0
        n_qual_items = 0
        qual_log: List[Dict[str, object]] = []

        for name, val in ql.factors_t0.items():
            # Skip missing / NaN values
            if val is None or (isinstance(val, float) and math.isnan(val)):
                logger.info(
                    "%s-Qual: factor %s has NaN/None value, skipping",
                    self.cp_name,
                    name,
                )
                continue

            s = score_qual_factor_numeric(val)           # Map 1–5 → 0–100 via QUAL_SCORE_SCALE
            if s is None:
                logger.info(
                    "%s-Qual: unknown or out-of-range factor %s=%s",
                    self.cp_name,
                    name,
                    val,
                )
                continue

            w = float(qual_weights.get(name, 1.0))       # Factor weight (default=1)
            bucket = qual_buckets.get(name, "OTHERS")    # Category for reporting

            total_weighted += w * s
            total_weight += w
            n_qual_items += 1

            logger.info(
                "%s-Qual: %s=%s score=%.1f weight=%.1f bucket=%s",
                self.cp_name,
                name,
                val,
                s,
                w,
                bucket,
            )
            qual_log.append(
                {"Name": name, "Value": val, "Score": s, "Weight": w, "Bucket": bucket}
            )

        qualitative_score = total_weighted / total_weight if total_weight > 0 else 0.0
        logger.info(
            "%s-Qual: aggregate weighted score=%.1f",
            self.cp_name,
            qualitative_score,
        )
        return qualitative_score, n_qual_items, qual_log

    # --------- DISTRESS / HARDSTOPS ---------

    def compute_distress_notches(
        self,
        fin: Dict[str, float],
    ) -> Tuple[int, Dict[str, float], Dict[str, int]]:
        total_notches = 0
        details: Dict[str, float] = {}
        per_metric_notches: Dict[str, int] = {}
    
        ic = fin.get("interest_coverage")
        if ic is not None:
            for threshold, notches in DISTRESS_BANDS["interest_coverage"]:
                if ic < threshold:
                    total_notches += notches
                    details["interest_coverage"] = ic
                    per_metric_notches["interest_coverage"] = notches
                    break
    
        dscr = fin.get("dscr")
        if dscr is not None:
            for threshold, notches in DISTRESS_BANDS["dscr"]:
                if dscr < threshold:
                    total_notches += notches
                    details["dscr"] = dscr
                    per_metric_notches["dscr"] = notches
                    break
    
        altman_z = fin.get("altman_z")
        if altman_z is not None:
            for threshold, notches in DISTRESS_BANDS["altman_z"]:
                if altman_z < threshold:
                    total_notches += notches
                    details["altman_z"] = altman_z
                    per_metric_notches["altman_z"] = notches
                    break
        # Cap maximum downgrade at 4 notches (notches are negative)
        if total_notches < MAX_DISTRESS_NOTCHES:
            total_notches = MAX_DISTRESS_NOTCHES
    
        return total_notches, details, per_metric_notches

    # --------- TOP-LEVEL ORCHESTRATION ---------

    def compute_final_rating(
        self,
        quant_inputs: QuantInputs,
        qual_inputs: QualInputs,
        sovereign_rating: Optional[str] = None,
        sovereign_outlook: Optional[str] = None,
        enable_hardstops: bool = False,
        enable_sovereign_cap: bool = False,
        enable_peer_positioning: bool = False,
        ratio_weights: Optional[Dict[str, float]] = None,
        qual_weights: Optional[Dict[str, float]] = None,
        qual_buckets: Optional[Dict[str, str]] = None,
    ) -> RatingOutputs:
        """Main API: returns fully populated RatingOutputs dataclass."""
        
        ratio_weights = ratio_weights or {}
        qual_weights = qual_weights or {}
        qual_buckets = qual_buckets or {}

        fin = dict(quant_inputs.fin_t0)

        # Altman once, on the same fin
        altman_z = self._ensure_altman_z(fin, quant_inputs.components_t0)
        
        # 1. Quantitative block
        (
            quant_score,
            peer_score,
            bucket_avgs,
            n_quant,
            peer_under,
            peer_over,
            peer_on_par,
            peer_total,
            ratio_log,
        ) = self.compute_quantitative(
            quant_inputs,
            ratio_weights=ratio_weights,
            enable_peer_positioning=enable_peer_positioning,
            fin=fin,
        )

        # 2. Qualitative block
        qual_score, n_qual, qual_log = self.compute_qualitative(
            qual_inputs,
            qual_weights=qual_weights,
            qual_buckets=qual_buckets,
        )

        # 3. Weights between quant and qual
        wq, wl = compute_effective_weights(n_quant, n_qual)
        logger.info(
            "%s-Weights: n_quant=%d n_qual=%d -> wq=%.3f wl=%.3f",
            self.cp_name,
            n_quant,
            n_qual,
            wq,
            wl,
        )

        combined_score = wq * quant_score + wl * qual_score

        # 4. Base rating and band-based outlook
        base_rating = safe_score_to_rating(combined_score)
        base_outlook, band_position = derive_outlook_band_only(combined_score, base_rating)

        # 5. Distress / hardstops
        if enable_hardstops:
            distress_notches, hardstop_details, distress_per_metric = self.compute_distress_notches(fin,)
        else:
            distress_notches = 0
            hardstop_details = {}
            distress_per_metric = {}

        # Enrich ratio_log with distress per metric
        for row in ratio_log:
            name = row.get("Name")
            if name in {"interest_coverage", "dscr", "altman_z"}:
                row["DistressNotches"] = distress_per_metric.get(name, 0)
            else:
                row["DistressNotches"] = row.get("DistressNotches", 0)

        hardstop_rating = move_notches(base_rating, distress_notches)
        hardstop_triggered = (hardstop_rating != base_rating)

        # 6. Sovereign cap
        capped_rating = hardstop_rating
        if enable_sovereign_cap and sovereign_rating is not None:
            capped_rating = apply_sovereign_cap(hardstop_rating, sovereign_rating)

        final_rating = capped_rating

        sovereign_cap_binding = (
            enable_sovereign_cap
            and sovereign_rating is not None
            and final_rating == sovereign_rating
        )

        # 7. Outlook adjustments (distress trend + sovereign)
        # If no hardstops are triggered, hardstop_outlook ≈ base_outlook; outlook follows a waterfall: base → hardstop → sovereign.
        hardstop_outlook = derive_outlook_with_distress_trend(
            base_outlook,
            distress_notches,
            quant_inputs.fin_t0,
            quant_inputs.fin_t1,
        )

        if (
            sovereign_cap_binding
            and sovereign_outlook in {"Positive", "Stable", "Negative"}
        ):
            sr = sovereign_rating
            so = sovereign_outlook
            severity = {"Positive": 0, "Stable": 1, "Negative": 2}

            if is_stronger(hardstop_rating, sr):
                outlook = so                                  # Sovereign stronger → use its outlook
            else:
                if sr == hardstop_rating:
                    candidates = [hardstop_outlook, so]
                    outlook = max(candidates, key=lambda o: severity[o])
        else:
            outlook = hardstop_outlook

        if final_rating == "AAA" and outlook == "Positive" and not sovereign_cap_binding:
            outlook = "Stable"                               # No AAA with Positive in this model

        final_outlook = outlook

        # 8. Flags for UI / debug
        flags = {
            "enable_hardstops": enable_hardstops,
            "enable_sovereign_cap": enable_sovereign_cap and (sovereign_rating is not None),
            "enable_peer_positioning": enable_peer_positioning,
            "hardstop_triggered": hardstop_triggered,
            "sovereign_cap_binding": sovereign_cap_binding,
        }

        # 9. Human-readable explanation string
        parts: List[str] = []
        parts.append(
            f"Based on the quantitative and qualitative factors, the combined score is "
            f"{combined_score:.1f}, corresponding to a base rating of {base_rating}."
        )
        if hardstop_triggered:
            parts.append(
                f" Distress factors {list(hardstop_details.keys())} triggered a total "
                f"of {abs(distress_notches)} notch(es) of downgrade, resulting in a "
                f"post-distress (hardstop) rating of {hardstop_rating}."
            )
        else:
            parts.append(
                f" No distress-related hardstops were applied, so the hardstop rating "
                f"remains equal to the base rating at {hardstop_rating}."
            )

        if enable_sovereign_cap and sovereign_rating is not None:
            if sovereign_cap_binding:
                if hardstop_rating != capped_rating:
                    parts.append(
                        f" The sovereign cap is binding: given the sovereign rating of "
                        f"{sovereign_rating}, the rating is constrained from {hardstop_rating} "
                        f"to a capped rating of {capped_rating}."
                    )
                else:
                    parts.append(
                        f" The issuer's rating is aligned with the sovereign rating at "
                        f"{sovereign_rating}, so the sovereign cap is effectively binding."
                    )
            else:
                parts.append(
                    f" A sovereign rating of {sovereign_rating} is considered, but it does not "
                    f"constrain the issuer rating, so the capped rating remains {capped_rating}."
                )
        else:
            parts.append(
                f" No sovereign cap is applied, so the capped rating is the same as the "
                f"post-distress rating at {capped_rating}."
            )

        parts.append(
            f" The final issuer rating is {final_rating} with an outlook of {final_outlook}."
        )
        rating_explanation = "".join(parts)

        logger.info(
            "%s-Final: base=%s hardstop=%s capped=%s final=%s outlook=%s distress_notches=%d",
            self.cp_name,
            base_rating,
            hardstop_rating,
            capped_rating,
            final_rating,
            final_outlook,
            distress_notches,
        )

        # 10. Return fully populated RatingOutputs dataclass
        return RatingOutputs(
            issuer_name=self.cp_name,
            quantitative_score=quant_score,
            qualitative_score=qual_score,
            combined_score=combined_score,
            peer_score=peer_score,
            base_rating=base_rating,
            distress_notches=distress_notches,
            hardstop_rating=hardstop_rating,
            capped_rating=capped_rating,
            final_rating=final_rating,
            hardstop_triggered=hardstop_triggered,
            hardstop_details=hardstop_details,
            sovereign_rating=sovereign_rating,
            sovereign_outlook=sovereign_outlook,
            sovereign_cap_binding=sovereign_cap_binding,
            outlook=final_outlook,
            bucket_avgs=bucket_avgs,
            altman_z_t0=fin.get("altman_z"),
            flags=flags,
            rating_explanation=rating_explanation,
            peer_underperform_count=peer_under,
            peer_outperform_count=peer_over,
            peer_on_par_count=peer_on_par,
            peer_total_compared=peer_total,
            band_position=band_position,
            ratio_log=ratio_log,
            base_outlook=base_outlook,
            hardstop_outlook=hardstop_outlook,
            final_outlook=final_outlook,
            n_quant=n_quant,
            n_qual=n_qual,
            wq=wq,
            wl=wl,
            qual_log=qual_log,
        )

## 11. Excel rating report generator

The `generate_corporate_rating_report` function builds a formatted multi-sheet Excel report from a `RatingOutputs` object and the original input workbook. It reloads the input ratios, qualitative factors, metadata, and peers from `sn_rating_input.xlsx`, computes peer averages for display, and then creates a main “Rating Report” sheet with issuer header details, key quantitative and qualitative values, score and flag summaries, and a narrative rating rationale.It also generates a detailed “log” sheet that combines the per-ratio log with peer and distress information, plus an optional `qual_log` sheet with qualitative factor-level details, and finally saves the report to the output directory with an issuer-specific filename.

In [471]:
def generate_corporate_rating_report(result) -> str:
    """Create a formatted Excel report (main + log sheets) from RatingOutputs."""
    
    res = result  # Alias for brevity

    # ------------------------------------------------------------------
    # Load input sheets for display (only for showing original ratios/factors)
    # ------------------------------------------------------------------
    rating_input_file = input_path("sn_rating_input.xlsx")  # Input workbook path

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
        with pd.ExcelFile(rating_input_file) as xls:
            df_fin = pd.read_excel(xls, sheet_name="fin_ratios", index_col=0)
            df_qual = pd.read_excel(xls, sheet_name="qual_factors", index_col=0)
            df_meta = pd.read_excel(xls, sheet_name="metadata")
            df_peers = pd.read_excel(xls, sheet_name="peers_t0")

    # Drop unnamed helper columns
    df_fin = df_fin.loc[:, ~df_fin.columns.str.contains("^Unnamed")]
    df_qual = df_qual.loc[:, ~df_qual.columns.str.contains("^Unnamed")]

    YEAR_T0, YEAR_T1, YEAR_T2 = _infer_time_labels(df_fin)  # Infer T0/T1/T2 labels

    meta = dict(zip(df_meta["field"], df_meta["value"]))  # metadata → dict

    raw_name = str(meta.get("name", "")).strip()
    issuer_name = raw_name.title()
    sovereign_rating: Optional[str] = (meta.get("sovereign_rating") or "").strip() or None
    sovereign_outlook: Optional[str] = (meta.get("sovereign_outlook") or "").strip() or None

    enable_hardstops = res.flags.get("enable_hardstops", False)
    enable_sovereign_cap = res.flags.get("enable_sovereign_cap", False)
    enable_peer_positioning = res.flags.get("enable_peer_positioning", False)

    peer_avg_fin: Dict[str, float] = {}  # Peer averages for ratios
    peer_score = res.peer_score          # Already computed by model

    if enable_peer_positioning and not df_peers.empty:
        df_peers_indexed = df_peers.set_index(df_peers.columns[0])  # First col as index
        for ratio in df_fin.index:
            if ratio in df_peers_indexed.index:
                peer_avg_fin[ratio] = df_peers_indexed.loc[ratio].mean()

    # ------------------------------------------------------------------
    # Set up main workbook and sheet
    # ------------------------------------------------------------------
    wb = Workbook()
    ws = wb.active
    ws.title = "Rating Report"

    bold = Font(bold=True)
    title_font = Font(size=14, bold=True)

    # Column widths for main report
    widths = [23, 13, 13, 23, 13, 13]
    for i, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = w

    # Report title row
    ws.merge_cells("A1:F1")
    ws["A1"] = "CORPORATE CREDIT RATING REPORT"
    ws["A1"].font = title_font
    ws["A1"].alignment = Alignment(horizontal="center")

    report_date = datetime.today().strftime("%Y-%m-%d")
    run_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    internal_id = f"RR-{datetime.today().strftime('%Y%m%d')}"  # Simple daily ref

    header = [
        ("Issuer Name", issuer_name, "Internal Ref ID", internal_id),
        ("Report Date", report_date, "Run Timestamp", run_timestamp),
        ("Final Rating", res.final_rating, "Generated By", "SN Corporate Rating Model"),
        ("Outlook", res.final_outlook, "Model Version", "V1.0"),
    ]

    row = 3
    for left_label, left_val, right_label, right_val in header:
        ws[f"A{row}"] = left_label
        ws[f"A{row}"].font = bold
        ws[f"B{row}"] = left_val
        ws[f"D{row}"] = right_label
        ws[f"D{row}"].font = bold
        ws[f"E{row}"] = right_val
        row += 1

    row += 1  # blank row after header block

    # Section headers for ratios/factors
    ws[f"A{row}"] = "Quantitative ratios"
    ws[f"A{row}"].font = bold
    ws[f"B{row}"] = YEAR_T0
    ws[f"C{row}"] = YEAR_T1

    ws[f"D{row}"] = "Qualitative factors"
    ws[f"D{row}"].font = bold
    ws[f"E{row}"] = YEAR_T0
    ws[f"F{row}"] = YEAR_T1

    # Bold + center A..F (so FY24 on the qual side is included)
    for col in range(1, 7):
        cell = ws.cell(row=row, column=col)
        cell.font = bold
        cell.alignment = Alignment(horizontal="center")

    row += 1  # move to first data row

    # ------------------------------------------------------------------
    # Build ordered lists from logs (to choose which rows to show)
    # ------------------------------------------------------------------
    ratio_rows: List[tuple] = []
    for r in res.ratio_log:
        name = r.get("Name")
        if not name:
            continue
        if name not in df_fin.index:
            continue
        ratio_rows.append((name, r.get("Value")))
    fin_index = [name for name, _ in ratio_rows]

    qual_rows: List[str] = []
    for r in res.qual_log:
        name = r.get("Name")
        if not name:
            continue
        qual_rows.append(name)

    if qual_rows:
        qual_index = qual_rows
    else:
        qual_index = list(df_qual.index)

    table_start_row = row
    TABLE_ROWS = max(20, len(fin_index), len(qual_index))

    for i in range(TABLE_ROWS):
        if i < len(fin_index):
            ratio = fin_index[i]
            ws[f"A{row}"] = ratio
            ws[f"B{row}"] = df_fin.loc[ratio].get(YEAR_T0)
            ws[f"C{row}"] = df_fin.loc[ratio].get(YEAR_T1)
            for col in ("B", "C"):
                cell = ws[f"{col}{row}"]
                if isinstance(cell.value, (int, float)):
                    cell.number_format = "0.00"
                    cell.alignment = Alignment(horizontal="right")

        if i < len(qual_index):
            factor = qual_index[i]
            ws[f"D{row}"] = factor
            ws[f"E{row}"] = df_qual.loc[factor].get(YEAR_T0)
            ws[f"F{row}"] = df_qual.loc[factor].get(YEAR_T1)
            for col in ("E", "F"):
                cell = ws[f"{col}{row}"]
                if isinstance(cell.value, (int, float)):
                    cell.number_format = "0.00"
                    cell.alignment = Alignment(horizontal="right")

        row += 1

    table_end_row = row - 1

    row += 1  # blank row after ratios/factors table
    score_start = row

    # ------------------------------------------------------------------
    # Score summary block (right-hand side)
    # ------------------------------------------------------------------
    score_block = [
        ("Quantitative Score", res.quantitative_score),
        ("Qualitative Score", res.qualitative_score),
        ("Combined Score", res.combined_score),
        ("Base Rating/Outlook", f"{res.base_rating}/{res.base_outlook}"),
        ("Hard Stop Rating/Outlook", f"{res.hardstop_rating}/{res.hardstop_outlook}"),
        ("Final Rating/Outlook", f"{res.final_rating}/{res.final_outlook}"),
    ]

    for label, value in score_block:
        ws[f"D{row}"] = label
        ws[f"D{row}"].font = bold
        ws[f"F{row}"] = value
        row += 1

    score_block_end_row = row - 1

    # ------------------------------------------------------------------
    # Flags block on left side
    # ------------------------------------------------------------------
    flags = [
        ("Peer positioning enabled", enable_peer_positioning),
        ("Hardstop enabled",         enable_hardstops),
        ("Sovereign cap enabled",    enable_sovereign_cap),
        ("Hardstop triggered",       res.hardstop_triggered),
        ("Distress_notches",         res.distress_notches),
        ("Sovereign Rating/Outlook", f"{sovereign_rating}/{sovereign_outlook}"),
    ]

    for i, (label, value) in enumerate(flags):
        ws[f"A{score_start+i}"] = label
        ws[f"A{score_start+i}"].font = bold
        ws[f"C{score_start+i}"] = value

    flags_end_row = score_start + len(flags) - 1

    # ------------------------------------------------------------------
    # Rating rationale block
    # ------------------------------------------------------------------
    # One blank row between flags and RATING RATIONALE
    row = flags_end_row + 1  # empty spacer row
    row += 1                 # RATING RATIONALE title row
    rating_title_row = row

    ws[f"A{rating_title_row}"] = "RATING RATIONALE"
    ws[f"A{rating_title_row}"].font = bold

    row += 1                 # first line of text
    rationale_top = row
    rationale_height = 10
    rationale_bottom = rationale_top + rationale_height - 1

    ws.merge_cells(start_row=rationale_top, start_column=1,
                   end_row=rationale_bottom, end_column=6)
    ws[f"A{rationale_top}"] = res.rating_explanation
    ws[f"A{rationale_top}"].alignment = Alignment(wrap_text=True, vertical="top")

    # ------------------------------------------------------------------
    # Page setup (A4, fit to one page)
    # ------------------------------------------------------------------
    ws.page_setup.orientation = ws.ORIENTATION_PORTRAIT
    ws.page_setup.paperSize = ws.PAPERSIZE_A4
    ws.page_margins = PageMargins(
        left=0.3,
        right=0.3,
        top=0.5,
        bottom=0.5,
        header=0.2,
        footer=0.2,
    )
    ws.sheet_properties.pageSetUpPr.fitToPage = True
    ws.page_setup.fitToWidth = 1
    ws.page_setup.fitToHeight = 1
    last_row = rationale_bottom
    ws.print_area = f"A1:F{last_row}"

    # Align flags and score numbers to the right
    for r in range(score_start, score_start + len(flags)):
        ws[f"C{r}"].alignment = Alignment(horizontal="right")

    for r in range(score_start, score_start + len(score_block)):
        c = ws[f"F{r}"]
        c.alignment = Alignment(horizontal="right")
        if isinstance(c.value, (int, float)):
            c.number_format = "0.00"

    # ------------------------------------------------------------------
    # Borders
    # ------------------------------------------------------------------
    thick = Side(style="thick")

    def apply_outer_thick_border(ws_, rng: str) -> None:
        """Draw a thick border around a rectangular cell range."""
        min_col, min_row, max_col, max_row = range_boundaries(rng)
        for row_cells in ws_.iter_rows(
            min_row=min_row,
            max_row=max_row,
            min_col=min_col,
            max_col=max_col,
        ):
            for cell in row_cells:
                b = cell.border
                top = b.top
                bottom = b.bottom
                left = b.left
                right = b.right

                if cell.row == min_row:
                    top = thick
                if cell.row == max_row:
                    bottom = thick
                if cell.column == min_col:
                    left = thick
                if cell.column == max_col:
                    right = thick

                cell.border = Border(top=top, bottom=bottom, left=left, right=right)

    # Outer frame from title through rationale bottom (keep)
    table_frame_bottom = table_end_row + 1  # one row after the last ratio row
    score_frame_bottom = flags_end_row + 1  # one row after last flag row
    
    apply_outer_thick_border(ws, f"A1:F{rationale_bottom}")
    apply_outer_thick_border(ws, "A2:F7")
    apply_outer_thick_border(ws, f"A{table_start_row}:F{table_frame_bottom}")
    apply_outer_thick_border(ws, f"A{table_start_row}:C{table_frame_bottom}")
    apply_outer_thick_border(ws, f"A{score_start}:F{score_frame_bottom}")
    apply_outer_thick_border(ws, f"A{score_start}:C{score_frame_bottom}")

    # ------------------------------------------------------------------
    # Log sheet (ratio log + peer/distress summary)
    # ------------------------------------------------------------------
    log_ws = wb.create_sheet(title="log")

    log_rows: List[Dict[str, Any]] = []
    log_rows.extend(res.ratio_log)                         # Start with detailed ratio log

    # Derive peer counts from PeerFlag in ratio_log
    under_count = sum(1 for r in res.ratio_log if r.get("PeerFlag") == "under")
    over_count = sum(1 for r in res.ratio_log if r.get("PeerFlag") == "over")
    on_par_count = sum(1 for r in res.ratio_log if r.get("PeerFlag") == "on_par")
    total_count = under_count + over_count + on_par_count

    # Blank separator
    log_rows.append({"Name": "", "Value": "", "Score": "", "PeerFlag": None})
    
    peer_score = res.peer_score
    # Under / total share (simple peer underperformance share)
    if total_count > 0:
        under_share = under_count / total_count
    else:
        under_share = None

    for row_dict in log_rows:
        if row_dict.get("Name") == "peer_positioning":
            row_dict["Value"] = peer_score
            break

    # Summary rows based on flags/aggregates
    log_rows.append(
        {
            "Name": "Peer positioning enabled",
            "Value": res.flags.get("enable_peer_positioning", False),
            "Score": "",
            "PeerFlag": None,
        }
    )
    log_rows.append(
        {
            "Name": "peer_underperform_count",
            "Value": under_count,
            "Score": "",
            "PeerFlag": None,
        }
    )
    log_rows.append(
        {
            "Name": "peer_outperform_count",
            "Value": over_count,
            "Score": "",
            "PeerFlag": None,
        }
    )
    log_rows.append(
        {
            "Name": "peer_on_par_count",
            "Value": on_par_count,
            "Score": "",
            "PeerFlag": None,
        }
    )
    log_rows.append(
        {
            "Name": "peer_total_compared",
            "Value": total_count,
            "Score": "",
            "PeerFlag": None,
        }
    )
    log_rows.append(
        {
            "Name": "peer_positioning",
            "Value": under_share,   
            "Score": res.peer_score,
            "PeerFlag": None,
        }
    )
    log_rows.append(
        {
            "Name": "aggregate quantitative score",
            "Value": "",
            "Score": res.quantitative_score,
            "PeerFlag": None,
        }
    )
    log_rows.append(
        {
            "Name": "aggregate qualitative score",
            "Value": "",
            "Score": res.qualitative_score,
            "PeerFlag": None,
        }
    )

    log_rows.append(
        {
            "Name": "n_quant/weights",
            "Value": getattr(res, "n_quant", None),
            "Score": getattr(res, "wq", None),
            "PeerFlag": None,
        }
    )
    log_rows.append(
        {
            "Name": "n_qual/weights",
            "Value": getattr(res, "n_qual", None),
            "Score": getattr(res, "wl", None),
            "PeerFlag": None,
        }
    )

    log_rows.append(
        {
            "Name": "combined_score",
            "Value": res.combined_score,
            "Score": "",
            "PeerFlag": None,
        }
    )

    band_min, band_max = get_rating_band(res.base_rating)
    band_range_str = f"{int(band_min)}-{int(band_max)}"
    log_rows.append(
        {
            "Name": "band_position",
            "Value": band_range_str,
            "Score": res.band_position,
            "PeerFlag": None,
        }
    )

    log_rows.append(
        {
            "Name": "base_rating/outlook",
            "Value": f"{res.base_rating}/{res.base_outlook}",
            "Score": "",
            "PeerFlag": None,
        }
    )
    log_rows.append(
        {
            "Name": "hardstop_rating/outlook",
            "Value": f"{res.hardstop_rating}/{res.hardstop_outlook}",
            "Score": "",
            "PeerFlag": None,
        }
    )
    log_rows.append(
        {
            "Name": "final rating/outlook",
            "Value": f"{res.final_rating}/{res.final_outlook}",
            "Score": "",
            "PeerFlag": None,
        }
    )

    log_rows.append(
        {
            "Name": "Hardstop enabled",
            "Value": res.flags.get("enable_hardstops", False),
            "Score": "",
            "PeerFlag": None,
        }
    )
    log_rows.append(
        {
            "Name": "Sovereign cap enabled",
            "Value": res.flags.get("enable_sovereign_cap", False),
            "Score": "",
            "PeerFlag": None,
        }
    )
    log_rows.append(
        {
            "Name": "distress_notches",
            "Value": res.distress_notches,
            "Score": "",
            "PeerFlag": None,
        }
    )
    log_rows.append(
        {
            "Name": "sovereign Rating/Outlook",
            "Value": f"{res.sovereign_rating or ''}/{res.sovereign_outlook or ''}",
            "Score": "",
            "PeerFlag": None,
        }
    )

    # Build DataFrame and preserve peer/distress columns
    log_df = pd.DataFrame(log_rows)

    # Ensure base columns exist
    for col in [
        "Name",
        "Value",
        "Score",
        "PeerFlag",
        "PeerAvg",
        "PeerLowerBound",
        "PeerUpperBound",
        "DistressNotches",
    ]:
        if col not in log_df.columns:
            log_df[col] = None

    log_df = log_df[
        [
            "Name",
            "Value",
            "Score",
            "PeerFlag",
            "PeerAvg",
            "PeerLowerBound",
            "PeerUpperBound",
            "DistressNotches",
        ]
    ]

    # Write header row
    bold = Font(bold=True)
    for j, col in enumerate(log_df.columns, start=1):
        log_ws.cell(row=1, column=j, value=col).font = bold

    # Write data rows with number formats and alignment
    for i, (_, row_series) in enumerate(log_df.iterrows(), start=2):
        name = row_series["Name"]

        log_ws.cell(row=i, column=1, value=name)

        v_cell = log_ws.cell(row=i, column=2, value=row_series["Value"])
        s_cell = log_ws.cell(row=i, column=3, value=row_series["Score"])
        p_cell = log_ws.cell(row=i, column=4, value=row_series["PeerFlag"])
        avg_cell = log_ws.cell(row=i, column=5, value=row_series["PeerAvg"])
        lb_cell = log_ws.cell(row=i, column=6, value=row_series["PeerLowerBound"])
        ub_cell = log_ws.cell(row=i, column=7, value=row_series["PeerUpperBound"])
        dn_cell = log_ws.cell(row=i, column=8, value=row_series["DistressNotches"])

        # Number formatting
        if isinstance(row_series["Value"], (int, float)):
            if name in {
                "peer_underperform_count",
                "peer_outperform_count",
                "peer_on_par_count",
                "peer_total_compared",
                "n_quant/weights",
                "n_qual/weights",
                "distress_notches",
            }:
                v_cell.number_format = "0"
            else:
                v_cell.number_format = "0.00"

        if isinstance(row_series["Score"], (int, float)):
            s_cell.number_format = "0.00"
        if isinstance(row_series["PeerAvg"], (int, float)):
            avg_cell.number_format = "0.00"
        if isinstance(row_series["PeerLowerBound"], (int, float)):
            lb_cell.number_format = "0.00"
        if isinstance(row_series["PeerUpperBound"], (int, float)):
            ub_cell.number_format = "0.00"
        if isinstance(row_series["DistressNotches"], (int, float)):
            dn_cell.number_format = "0"

        # Right-align numeric cells
        for cell in (v_cell, s_cell, p_cell, avg_cell, lb_cell, ub_cell, dn_cell):
            cell.alignment = Alignment(horizontal="right")
       
    # Column widths for log sheet
    for col_letter, width in zip(
        ["A", "B", "C", "D", "E", "F", "G", "H"],
        [30, 20, 15, 12, 15, 15, 15, 15],
    ):
        log_ws.column_dimensions[col_letter].width = width

    # ------------------------------------------------------------------
    # Optional: qualitative log sheet
    # ------------------------------------------------------------------
    if res.qual_log:          # will be False only if the list is empty
        qual_ws = wb.create_sheet(title="qual_log")
        qual_df = pd.DataFrame(res.qual_log)

        # Ensure standard columns
        for col in ["Name", "Value", "Score", "Weight", "Bucket"]:
            if col not in qual_df.columns:
                qual_df[col] = None
        qual_df = qual_df[["Name", "Value", "Score", "Weight", "Bucket"]]

        for j, col in enumerate(qual_df.columns, start=1):
            qual_ws.cell(row=1, column=j, value=col).font = bold

        for i, (_, row_series) in enumerate(qual_df.iterrows(), start=2):
            qual_ws.cell(row=i, column=1, value=row_series["Name"])
            v_cell = qual_ws.cell(row=i, column=2, value=row_series["Value"])
            s_cell = qual_ws.cell(row=i, column=3, value=row_series["Score"])
            w_cell = qual_ws.cell(row=i, column=4, value=row_series["Weight"])
            b_cell = qual_ws.cell(row=i, column=5, value=row_series["Bucket"])

            if isinstance(row_series["Value"], (int, float)):
                v_cell.number_format = "0"
            if isinstance(row_series["Score"], (int, float)):
                s_cell.number_format = "0.00"
            if isinstance(row_series["Weight"], (int, float)):
                w_cell.number_format = "0.00"

            for cell in (v_cell, s_cell, w_cell):
                cell.alignment = Alignment(horizontal="right")

        for col_letter, width in zip(["A", "B", "C", "D", "E"], [30, 15, 15, 15, 20]):
            qual_ws.column_dimensions[col_letter].width = width

    # ------------------------------------------------------------------
    # Save workbook to output folder
    # ------------------------------------------------------------------
    safe_name = issuer_name.replace(" ", "_")
    out_file = output_path(f"{safe_name}_Corporate_Credit_Rating_Report.xlsx")

    wb.save(out_file)
    return out_file

## 12. Excel-driven model runner

The `run_from_excel_with_bands` function is the high-level entry point that runs the V3 rating model directly from the Excel input files. It loads the band configuration and user-edited inputs (`sn_rating_config.xlsx`, `sn_rating_input.xlsx`), cleans the DataFrames, infers time labels, and converts financial ratios, Altman Z components, qualitative factors, and peer data into the corresponding `QuantInputs` and `QualInputs` dataclasses. It then instantiates `RatingModel` with the issuer name and band configuration, applies feature flags and sovereign settings from metadata, executes `compute_final_rating`, and returns a fully populated `RatingOutputs` object for further use (e.g. report generation).


In [474]:
def run_from_excel_with_bands(
    horizon: Literal["t0", "t1", "t2"] = "t0",
):
    """
    High-level runner for the V3 model.

    Excel files are taken from the project root input folder:
      - input/sn_rating_config.xlsx (bands/config)
      - input/sn_rating_input.xlsx (user-edited input)
    """
    # Resolve input file paths
    rating_input_file = input_path("sn_rating_input.xlsx")
    config_file = input_path("sn_rating_config.xlsx")

    # Load band configuration (score tables) and global metadata
    bands = BandConfig(config_path=config_file)
    meta = load_metadata_excel(rating_input_file)

    raw_name = str(meta.get("name", "")).strip()
    cp_name = raw_name.title()                     # Title-case issuer name
    meta["name"] = cp_name                         # Normalize in metadata

    # Read all input sheets with warnings suppressed for openpyxl
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
        with pd.ExcelFile(rating_input_file) as xls:
            df_fin = pd.read_excel(xls, sheet_name="fin_ratios")
            df_comp = pd.read_excel(xls, sheet_name="components", index_col=0)
            df_qual = pd.read_excel(xls, sheet_name="qual_factors")
            df_peers_raw = pd.read_excel(xls, sheet_name="peers_t0")

    # Drop helper/empty 'Unnamed' columns
    df_fin = df_fin.loc[:, ~df_fin.columns.str.contains("^Unnamed")]
    df_qual = df_qual.loc[:, ~df_qual.columns.str.contains("^Unnamed")]
    df_peers_raw = df_peers_raw.loc[:, ~df_peers_raw.columns.str.contains("^Unnamed")]

    # ---------- FINANCIAL RATIOS (metric, weight, FYxx...) ----------
    YEAR_T0, YEAR_T1, YEAR_T2 = _infer_time_labels(
        df_fin,
        drop_cols=["metric", "weight"],
    )

    fin_t0 = {row["metric"]: row[YEAR_T0] for _, row in df_fin.iterrows()}
    fin_t1 = {row["metric"]: row[YEAR_T1] for _, row in df_fin.iterrows()}
    fin_t2 = {row["metric"]: row[YEAR_T2] for _, row in df_fin.iterrows()}

    ratio_weights = {
        row["metric"]: (row["weight"] if pd.notna(row.get("weight")) else 1.0)
        for _, row in df_fin.iterrows()
    }                                              # Per-ratio weight (default 1.0)

    # ---------- ALTMAN COMPONENTS ----------
    comp_t0 = components_col_to_dict(df_comp, YEAR_T0)  # Map code → value
    comp_t1 = components_col_to_dict(df_comp, YEAR_T1)
    comp_t2 = components_col_to_dict(df_comp, YEAR_T2)

    # ---------- QUAL FACTORS (factor, weight, bucket, FYxx...) ----------
    YEAR_Q0, YEAR_Q1, YEAR_Q2 = _infer_time_labels(
        df_qual,
        drop_cols=["factor", "weight", "bucket"],
    )

    qual_t0 = {row["factor"]: row[YEAR_Q0] for _, row in df_qual.iterrows()}
    qual_t1 = {row["factor"]: row[YEAR_Q1] for _, row in df_qual.iterrows()}

    qual_weights = {
        row["factor"]: (row["weight"] if pd.notna(row.get("weight")) else 1.0)
        for _, row in df_qual.iterrows()
    }
    qual_buckets = {
        row["factor"]: (
            row["bucket"]
            if pd.notna(row.get("bucket")) and str(row["bucket"]).strip()
            else "OTHERS"
        )
        for _, row in df_qual.iterrows()
    }

    # ---------- PEERS: use all peer columns after first (metric) ----------
    if not df_peers_raw.empty:
        first_col = df_peers_raw.columns[0]
        df_peers_raw.rename(columns={first_col: "metric"}, inplace=True)
        df_peers_raw.set_index("metric", inplace=True)
        df_peers = df_peers_raw                             # Now index = metric code
    else:
        df_peers = pd.DataFrame()

    peers_t0 = peers_df_to_dict(df_peers) if not df_peers.empty else {}  # "metric" → [values]

    # ---------- BUILD INPUT DATACLASSES ----------
    q_inputs = QuantInputs(
        fin_t0=fin_t0,
        fin_t1=fin_t1,
        fin_t2=fin_t2,
        components_t0=comp_t0,
        components_t1=comp_t1,
        components_t2=comp_t2,
        peers_t0=peers_t0,
    )

    qual_inputs = QualInputs(
        factors_t0=qual_t0,
        factors_t1=qual_t1,
    )

    # Instantiate model with issuer name and band config
    model = RatingModel(cp_name, bands)

    # Feature flags from metadata (with sane defaults)
    enable_hardstops = meta["enable_hardstops"]
    enable_sovereign_cap = meta["enable_sovereign_cap"]
    enable_peer_positioning = meta["enable_peer_positioning"]

    # Sovereign inputs: clean empty strings → None
    sovereign_rating: Optional[str] = (meta.get("sovereign_rating") or "").strip() or None
    sovereign_outlook: Optional[str] = (meta.get("sovereign_outlook") or "").strip() or None

    # ---------- RUN MODEL ----------
    res = model.compute_final_rating(
        quant_inputs=q_inputs,
        qual_inputs=qual_inputs,
        sovereign_rating=sovereign_rating,
        sovereign_outlook=sovereign_outlook,
        enable_hardstops=enable_hardstops,
        enable_sovereign_cap=enable_sovereign_cap,
        enable_peer_positioning=enable_peer_positioning,
        ratio_weights=ratio_weights,
        qual_weights=qual_weights,
        qual_buckets=qual_buckets,
    )

    return res                                     # RatingOutputs dataclass

## 13. Notebook runner and CLI-style utilities  

This section adapts file-path helpers for the notebook environment and provides simple console views of the model output. 

- **13.1 project_root / input_dir / output_dir / input_path / output_path / resource_path** – Notebook-friendly path helpers that treat the current working directory as the project root and ensure `input` and `output` folders exist, replacing the package/EXE-specific logic.

In [477]:
def project_root() -> str:
    """
    Notebook version: treat current working directory as project root.
    Ensure cwd contains the 'input' and 'output' folders.
    """
    return os.getcwd()

In [478]:
def input_dir() -> str:
    root = project_root()
    path = os.path.join(root, "input")
    os.makedirs(path, exist_ok=True)
    return path

In [479]:
def output_dir() -> str:
    root = project_root()
    path = os.path.join(root, "output")
    os.makedirs(path, exist_ok=True)
    return path

In [480]:
def input_path(name: str) -> str:
    return os.path.join(input_dir(), name)

In [481]:
def output_path(name: str) -> str:
    return os.path.join(output_dir(), name)

In [482]:
def resource_path(relative_path: str) -> str:
    """
    Notebook version: resolve resources relative to cwd/project_root.
    This replaces the __file__/sys.frozen logic.
    """
    base = project_root()  # in the notebook, project_root() should be cwd-based
    return os.path.join(base, relative_path)


- **13.2 print_ratio_log_cli** – Pretty-prints the T0 quantitative ratio log to the console, including value, score, weight, peer band (average, bounds, flag), and distress notches, for quick inspection without opening Excel.

In [484]:
def print_ratio_log_cli(res) -> None:
    """Pretty-print quantitative ratio log for T0 to the console, including peer band and distress."""

    print("\n=== Ratio log (T0) ===")
    header = (
        f"{'Name':25} "
        f"{'Value':>10} "
        f"{'Score':>8} "
        f"{'Weight':>8} "
        f"{'PeerAvg':>10} "
        f"{'PeerLowerBound':>15} "
        f"{'PeerUpperBound':>15} "
        f"{'PeerFlag':>10} "
        f"{'DistressNotches':>16}"
    )
    print(header)
    print("-" * len(header))

    for row in res.ratio_log:
        name = str(row.get("Name", ""))
        value = float(row.get("Value", 0.0) or 0.0)
        score = float(row.get("Score", 0.0) or 0.0)
        weight = float(row.get("Weight", 0.0) or 0.0)

        peer_avg = float(row.get("PeerAvg", 0.0) or 0.0)
        peer_low = float(row.get("PeerLowerBound", 0.0) or 0.0)
        peer_high = float(row.get("PeerUpperBound", 0.0) or 0.0)

        peer_flag = str(row.get("PeerFlag", row.get("PeerFlg", "")) or "")
        distress = int(row.get("DistressNotches", row.get("Distress", 0)) or 0)

        print(
            f"{name:25} "
            f"{value:10.2f} "
            f"{score:8.2f} "
            f"{weight:8.2f} "
            f"{peer_avg:10.2f} "
            f"{peer_low:15.2f} "
            f"{peer_high:15.2f} "
            f"{peer_flag:10} "
            f"{distress:16d}"
        )


- **13.3 print_qual_log_cli** – Pretty-prints the T0 qualitative factors log to the console, showing factor name, selected value, score, weight, and bucket.

In [486]:
def print_qual_log_cli(res) -> None:
    """Pretty-print qualitative factors log for T0 to the console."""

    print("\n=== Qualitative factors (T0) ===")
    header = (
        f"{'Name':35} {'Value':>8} {'Score':>8} {'Weight':>8} {'Bucket':>12}"
    )
    print(header)
    print("-" * len(header))

    for row in res.qual_log:
        name = str(row.get("Name", ""))
        value = int(row.get("Value", 0) or 0)
        score = float(row.get("Score", 0.0) or 0.0)
        weight = float(row.get("Weight", 0.0) or 0.0)
        bucket = str(row.get("Bucket", "") or "")

        print(
            f"{name:35} "
            f"{value:8d} "
            f"{score:8.2f} "
            f"{weight:8.2f}"
            f"{bucket:>12}"
        )

- **13.4 Notebook main runner** – Calls `run_from_excel_with_bands()` to execute the model, generates the Excel rating report via `generate_corporate_rating_report`, prints ratio and qualitative logs to the console, and outputs the generated report path so users can open it from the `output` folder.

In [488]:
res = run_from_excel_with_bands()
out_file = generate_corporate_rating_report(res)
print_ratio_log_cli(res)
print_qual_log_cli(res)
print(out_file)


2026-03-18 23:15:06,968 - INFO - sn_rating - BandConfig: loaded families/directions for ratios: {'altman_z': 'altman', 'capex_dep': 'other', 'current_ratio': 'other', 'debt_capital': 'leverage', 'debt_ebitda': 'leverage', 'debt_equity': 'leverage', 'dscr': 'coverage', 'ebit_margin': 'profit', 'ebitda_margin': 'profit', 'fcf_debt': 'leverage_rev', 'ffo_debt': 'leverage_rev', 'fixed_charge_coverage': 'coverage', 'interest_coverage': 'coverage', 'nan': 'nan', 'net_debt_ebitda': 'leverage', 'quick_ratio': 'other', 'ratio_a': 'other', 'ratio_b': 'other', 'ratio_c': 'other', 'ratio_d': 'other', 'roa': 'profit', 'roe': 'profit', 'rollover_coverage': 'other'}
2026-03-18 23:15:07,045 - INFO - sn_rating - Nan-Quant: altman_z value=3.00 score=100.0 weight=1.0 family=altman
2026-03-18 23:15:07,048 - INFO - sn_rating - Nan-Quant: debt_ebitda value=1.90 score=100.0 weight=2.0 family=leverage
2026-03-18 23:15:07,050 - INFO - sn_rating - Nan-Quant: net_debt_ebitda value=1.40 score=100.0 weight=2.0 fam


=== Ratio log (T0) ===
Name                           Value    Score   Weight    PeerAvg  PeerLowerBound  PeerUpperBound   PeerFlag  DistressNotches
-----------------------------------------------------------------------------------------------------------------------------
altman_z                        3.00   100.00     1.00       1.31            1.18            1.45 over                      0
debt_ebitda                     1.90   100.00     2.00       2.94            2.65            3.23 over                      0
net_debt_ebitda                 1.40   100.00     2.00       2.72            2.45            3.00 over                      0
debt_equity                     0.70    75.00     2.00       1.41            1.27            1.55 over                      0
debt_capital                    0.40    50.00     2.00       0.51            0.46            0.56 over                      0
ffo_debt                        0.80   100.00     2.00       0.22            0.20            0